# Local Manager

Esta é uma documentação detalhada e didática da classe `LocalManager`, projetada para servir como um guia tanto para usuários quanto para desenvolvedores que desejam entender a estrutura deste gerenciador de dados NoSQL local.

---

## Visão Geral

O `LocalManager` é uma solução leve e eficiente para persistência de dados em formato NoSQL, utilizando o sistema de arquivos local como armazenamento. Ele emula o comportamento de bancos de dados modernos (como o MongoDB), organizando as informações em uma hierarquia de **Databases** (diretórios) e **Collections** (arquivos JSON).

É a ferramenta ideal para:

- Prototipagem rápida de aplicações.
- Armazenamento de configurações locais.
- Ambientes de teste onde não se deseja configurar um servidor de banco de dados complexo.

## Fluxo de Execução

O funcionamento da classe segue um ciclo previsível de manipulação de I/O de arquivos:

1. **Inicialização:** O gerenciador verifica ou cria a pasta raiz (`data`).
2. **Mapeamento:** Ao solicitar uma operação, ele resolve o caminho físico: `base/database/collection.json`.
3. **Carregamento:** O arquivo JSON é lido e desserializado para uma lista de dicionários Python em memória.
4. **Processamento:** Filtros, inserções ou atualizações são aplicados nesta lista.
5. **Persistência:** A lista resultante é serializada de volta para o arquivo JSON, garantindo a integridade dos dados.

## Resumo dos Métodos

| **Método** | **Descrição Breve** |
| --- | --- |
| `save_payload` | Insere um novo registro com ID único e timestamp. |
| `fetch_documents` | Filtra e retorna documentos da coleção. |
| `update_documents` | Modifica campos de documentos existentes com base em filtros. |
| `delete_documents` | Remove registros específicos ou múltiplos da coleção. |
| `_get_collection_path` | (Interno) Resolve e cria o caminho de diretórios para o banco. |
| `_load_collection` | (Interno) Lê os dados do arquivo JSON com tratamento de erro. |
| `_save_collection` | (Interno) Grava os dados no disco com indentação legível. |
| `_match_filter` | (Interno) Compara documentos com critérios de busca. |

---

## Arquitetura e Insights

- **Abstração de ID e Tempo:** O sistema remove a carga do desenvolvedor de gerar IDs únicos (`uuid4`) e registrar momentos de criação/atualização, injetando metadados automaticamente.
- **Segurança de Escrita:** Ao utilizar `os.makedirs(exist_ok=True)`, o código evita erros comuns de "diretório não encontrado" durante a execução.
- **Simplicidade de Busca:** A filtragem é baseada em igualdade simples, o que torna a curva de aprendizado baixíssima para novos usuários.

---

## Classe LocalManager

### Descrição

Gerenciador de persistência local que utiliza arquivos JSON para simular um banco de dados NoSQL. Organiza os dados em estruturas de banco e coleção dentro do sistema de arquivos.

### Argumentos

- **base_path** (*str*): Nome do diretório raiz onde todos os dados serão salvos. Padrão: `"data"`.

---

### Métodos

### 1. save_payload

**Descrição:** Adiciona um novo documento ao banco de dados. O método gera automaticamente um campo `_id` único e um campo `_created_at`.

**Argumentos:**

- `database_name` (*str*): Nome da base de dados.
- `collection_name` (*str*): Nome da coleção.
- `payload` (*dict*): Os dados que você deseja salvar.

**Retornos:**

- `dict`: Um dicionário contendo o status da operação e o ID gerado.

**Raises:**

- `Exception`: Erros de permissão de escrita ou falha no sistema de arquivos.

**Exemplo:**

```bash
manager.save_payload("loja", "produtos", {"nome": "Teclado", "preco": 150.0})
```

---

### 2. fetch_documents

**Descrição:** Recupera uma lista de documentos que coincidem com os critérios informados.

**Argumentos:**

- `database_name` (*str*): Nome da base de dados.
- `collection_name` (*str*): Nome da coleção.
- `filter` (*dict*, opcional): Filtro de igualdade (ex: `{"status": "ativo"}`).
- `limit` (*int*): Limite máximo de resultados (0 para ilimitado).

**Retornos:**

- `List[Dict]`: Lista de documentos encontrados.

**Raises:**

- `json.JSONDecodeError`: Se o arquivo estiver corrompido.

**Exemplo:**

```bash
manager.fetch_documents("loja", "produtos", filter={"nome": "Teclado"})
```

---

### 3. update_documents

**Descrição:** Localiza documentos via filtro e atualiza seus valores com um novo conjunto de dados.

**Argumentos:**

- `database_name` (*str*): Nome da base de dados.
- `collection_name` (*str*): Nome da coleção.
- `filter` (*dict*): Critério para encontrar os documentos a serem editados.
- `new_values` (*dict*): Novos campos e valores a serem inseridos/alterados.
- `multi` (*bool*): Se `True`, atualiza todos os encontrados; se `False`, apenas o primeiro.

**Retornos:**

- `dict`: Estatísticas contendo `matched_count` e `modified_count`.

**Raises:**

- `IOError`: Erro ao tentar salvar as alterações no disco.

**Exemplo:**

```bash
manager.update_documents("loja", "produtos", {"nome": "Teclado"}, {"preco": 130.0})
```

---

### 4. delete_documents

**Descrição:** Remove documentos da coleção que correspondam ao filtro passado.

**Argumentos:**

- `database_name` (*str*): Nome da base de dados.
- `collection_name` (*str*): Nome da coleção.
- `filter` (*dict*): Critério de seleção para remoção.
- `multi` (*bool*): Se deve remover todos os correspondentes ou apenas o primeiro.

**Retornos:**

- `dict`: Confirmação com o número de itens deletados (`deleted_count`).

**Raises:**

- `Exception`: Falhas genéricas de acesso ao arquivo.

**Exemplo:**

```bash
manager.delete_documents("loja", "produtos", {"status": "esgotado"}, multi=True)
```
